In [1]:
from cmath import nan

import numpy as np
import yfinance as yf
import pandas as pd
import datetime as dt

import hist_volatility
from riskfree_and_spot import *
from hist_volatility import *
from blackscholes import *
from binomial import *
from imp_volatility import *

In [2]:
option_style = "american" #european / american
valuation_date = "2026-08-04" #YYYY-MM-DD
ticker = "MSFT"
expiration = "2026-08-05"
strike = 487.5
option_type = "Call" # call / put
volatility_method = "historical" #historical / (implied /not yet implemented/)
hist_vol_lookback = 1 #in years

#BINOMIAL
n_for_binomial = 1500 #t0 + n steps

In [3]:
spot_price = get_spot_price_data(ticker=ticker, valuation_date=valuation_date)
t_t_m = option_tenor_calc(val_date=valuation_date, exp_date=expiration)
risk_free_rate = get_risk_free_rate(val_date=valuation_date, exp_date=expiration)
vol = get_volatility(ticker=ticker, val_date=valuation_date, lookback=hist_vol_lookback)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [4]:
data = pd.read_csv("../data/option_data_calc.csv")
data

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration,tenor,rf_rate,imp_vol
0,MSFT260807C00270000,2026-08-03 14:47:55+00:00,270.0,216.18,228.35,232.20,0.000000,0.000000,3.0,5,3.689454,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
1,MSFT260807C00280000,2026-06-29 15:30:42+00:00,280.0,95.05,218.35,221.90,0.000000,0.000000,NaN,3,3.294924,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
2,MSFT260807C00290000,2026-06-29 15:30:44+00:00,290.0,86.80,208.35,212.20,0.000000,0.000000,NaN,3,3.298830,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
3,MSFT260807C00300000,2026-08-06 17:39:41+00:00,300.0,196.11,198.35,202.20,6.860001,3.624835,62.0,153,3.113283,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
4,MSFT260807C00305000,2026-08-06 17:39:41+00:00,305.0,191.13,193.30,196.90,113.710010,146.874200,64.0,28,2.814456,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622,MSFT281215C00710000,2026-08-06 16:39:59+00:00,710.0,55.75,56.00,59.50,1.650001,3.049911,3.0,317,0.387305,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.344011
1623,MSFT281215C00715000,2026-08-05 18:54:20+00:00,715.0,52.00,55.35,58.75,0.000000,0.000000,19.0,868,0.387923,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.345017
1624,MSFT281215C00720000,2026-08-06 19:26:47+00:00,720.0,55.45,54.30,57.35,3.250000,6.226054,89.0,904,0.386259,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.344173
1625,MSFT281215C00725000,2026-08-06 19:23:16+00:00,725.0,54.40,53.30,56.00,2.530003,4.877584,142.0,1780,0.384703,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.343435


In [5]:
mid_price_list = []

for i, row in data.iterrows():
    mid = (row["bid"] + row["ask"]) / 2
    mid_price_list.append(mid)

data["mid_price"] = mid_price_list

data

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration,tenor,rf_rate,imp_vol,mid_price
0,MSFT260807C00270000,2026-08-03 14:47:55+00:00,270.0,216.18,228.35,232.20,0.000000,0.000000,3.0,5,3.689454,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,230.275
1,MSFT260807C00280000,2026-06-29 15:30:42+00:00,280.0,95.05,218.35,221.90,0.000000,0.000000,NaN,3,3.294924,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,220.125
2,MSFT260807C00290000,2026-06-29 15:30:44+00:00,290.0,86.80,208.35,212.20,0.000000,0.000000,NaN,3,3.298830,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,210.275
3,MSFT260807C00300000,2026-08-06 17:39:41+00:00,300.0,196.11,198.35,202.20,6.860001,3.624835,62.0,153,3.113283,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,200.275
4,MSFT260807C00305000,2026-08-06 17:39:41+00:00,305.0,191.13,193.30,196.90,113.710010,146.874200,64.0,28,2.814456,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,195.100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622,MSFT281215C00710000,2026-08-06 16:39:59+00:00,710.0,55.75,56.00,59.50,1.650001,3.049911,3.0,317,0.387305,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.344011,57.750
1623,MSFT281215C00715000,2026-08-05 18:54:20+00:00,715.0,52.00,55.35,58.75,0.000000,0.000000,19.0,868,0.387923,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.345017,57.050
1624,MSFT281215C00720000,2026-08-06 19:26:47+00:00,720.0,55.45,54.30,57.35,3.250000,6.226054,89.0,904,0.386259,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.344173,55.825
1625,MSFT281215C00725000,2026-08-06 19:23:16+00:00,725.0,54.40,53.30,56.00,2.530003,4.877584,142.0,1780,0.384703,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.343435,54.650


In [7]:
no_arbitrage_flag_list = []

for i, row in data.iterrows():
    k = row["strike"]
    rf = row["rf_rate"]
    t = row["tenor"]
    mid = row["mid_price"]
    lower_bound = max(0, spot_price - (k * np.exp(-rf * t)))
    upper_bound = spot_price
    if mid > lower_bound and mid < upper_bound:
        arbitrage_flag = True
    else:
        arbitrage_flag = False
    no_arbitrage_flag_list.append(arbitrage_flag)

data["no_arbitrage_flag"] = no_arbitrage_flag_list

data


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration,tenor,rf_rate,imp_vol,mid_price,no_arbitrage_flag
0,MSFT260807C00270000,2026-08-03 14:47:55+00:00,270.0,216.18,228.35,232.20,0.000000,0.000000,3.0,5,3.689454,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,230.275,True
1,MSFT260807C00280000,2026-06-29 15:30:42+00:00,280.0,95.05,218.35,221.90,0.000000,0.000000,NaN,3,3.294924,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,220.125,True
2,MSFT260807C00290000,2026-06-29 15:30:44+00:00,290.0,86.80,208.35,212.20,0.000000,0.000000,NaN,3,3.298830,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,210.275,True
3,MSFT260807C00300000,2026-08-06 17:39:41+00:00,300.0,196.11,198.35,202.20,6.860001,3.624835,62.0,153,3.113283,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,200.275,True
4,MSFT260807C00305000,2026-08-06 17:39:41+00:00,305.0,191.13,193.30,196.90,113.710010,146.874200,64.0,28,2.814456,True,REGULAR,USD,2026-08-07,0.008219,0.037800,NaN,195.100,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622,MSFT281215C00710000,2026-08-06 16:39:59+00:00,710.0,55.75,56.00,59.50,1.650001,3.049911,3.0,317,0.387305,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.344011,57.750,True
1623,MSFT281215C00715000,2026-08-05 18:54:20+00:00,715.0,52.00,55.35,58.75,0.000000,0.000000,19.0,868,0.387923,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.345017,57.050,True
1624,MSFT281215C00720000,2026-08-06 19:26:47+00:00,720.0,55.45,54.30,57.35,3.250000,6.226054,89.0,904,0.386259,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.344173,55.825,True
1625,MSFT281215C00725000,2026-08-06 19:23:16+00:00,725.0,54.40,53.30,56.00,2.530003,4.877584,142.0,1780,0.384703,False,REGULAR,USD,2028-12-15,2.367123,0.042159,0.343435,54.650,True
